# Set Up the Thesis Project from GitHub

Run this notebook once in Google Colab before running the week notebooks. It clones the GitHub repository and copies the project folders into `MyDrive/Thesis`, which is the path used by the training and report notebooks.

If you publish the repository under a different owner or name, edit `REPO_URL` in the first code cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import subprocess

REPO_URL = 'https://github.com/HannanSeyfi/unlearning-thesis.git'
REPO_DIR = Path('/content/unlearning-thesis')
DRIVE_THESIS_DIR = Path('/content/drive/MyDrive/Thesis')

# Keep False to avoid overwriting existing Drive outputs. Set True to refresh files from GitHub.
OVERWRITE_EXISTING = False

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
DRIVE_THESIS_DIR.mkdir(parents=True, exist_ok=True)

print('Cloned repo to:', REPO_DIR)
print('Drive thesis folder:', DRIVE_THESIS_DIR)

In [ ]:
PROJECT_ITEMS = [
    'Week 1',
    'Week 2',
    'Week 3',
    'Week 3.5',
    'Week 4',
    'Week 4 - Joint Training Experiments',
    'Week 5',
    'reports',
    'analysis',
    'Tools',
]

def copy_tree_merge(src: Path, dst: Path, overwrite: bool = False):
    if not src.exists():
        print('Skipping missing source:', src)
        return {'copied': 0, 'skipped': 0}

    copied = 0
    skipped = 0
    for item in src.rglob('*'):
        relative = item.relative_to(src)
        target = dst / relative
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue

        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and not overwrite:
            skipped += 1
            continue

        shutil.copy2(item, target)
        copied += 1

    return {'copied': copied, 'skipped': skipped}

summary = {}
for item in PROJECT_ITEMS:
    result = copy_tree_merge(REPO_DIR / item, DRIVE_THESIS_DIR / item, OVERWRITE_EXISTING)
    summary[item] = result
    print(f"{item}: copied {result['copied']}, skipped {result['skipped']}")

print('Copy complete.')

In [ ]:
# Week 4 and Week 5 expect the strict Week 3.5 adapter at this run name.
# The repository preserves the same run under reference_successful_run.
week35_results = DRIVE_THESIS_DIR / 'Week 3.5' / 'results'
reference_run = week35_results / 'reference_successful_run'
expected_run = week35_results / 'qwen05_high_accuracy_baseline'

if reference_run.exists() and not expected_run.exists():
    shutil.copytree(reference_run, expected_run)
    print('Created compatibility copy:', expected_run)
elif expected_run.exists():
    print('Compatibility path already exists:', expected_run)
else:
    print('No Week 3.5 reference run found to copy. Run the Week 3.5 strict baseline notebook before Week 4 or Week 5.')

## Next

Open notebooks from `COLAB_NOTEBOOKS.md` in the GitHub repository. For a fresh run, start with Week 2 to regenerate the synthetic data, then run Week 3, Week 3.5, Week 4, and Week 5 in order.